# Diagnóstico predictivo: horizonte de dos meses

**Proyecto 3 — Petén.** Este análisis es exploratorio. No modifica `pipeline.py`, los umbrales de aceptación ni las versiones publicadas. Se simula la ausencia simultánea de los dos meses posteriores al corte.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

# Funciona si VS Code inicia el notebook desde la raiz o desde notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "optimizacion.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "optimizacion.py").exists():
    raise FileNotFoundError("No se encontro src/optimizacion.py. Abre el proyecto desde su carpeta raiz.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.optimizacion import ejecutar_horizonte_dos

DATA = ROOT / "data" / "processed"
OUTPUT = DATA / "optimizacion"
if not DATA.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de datos: {DATA}")

# Ejecuta la evaluacion y define las variables usadas por las graficas.
detalle, resumen = ejecutar_horizonte_dos(DATA, OUTPUT)
print("Repositorio:", ROOT)
print("Origenes:", pd.to_datetime(detalle["origen"]).dt.strftime("%Y-%m").unique().tolist())
print("Evaluaciones:", len(detalle))
display(resumen)


Repositorio: C:\Users\Emadlg\Desktop\UVG\proyecto3_Data
Origenes: ['2025-10']
Evaluaciones: 48


,metodo,n,MAE,RMSE
0,persistencia,12,7153.151151,8987.180255
1,media_movil,12,8475.815290,10198.575396
2,reparto_ultimo,12,10503.057961,12345.847508
3,reparto_mediano,12,13591.770437,15253.138321


## Interpretación

Con la cobertura observada de septiembre a diciembre de 2025, y dos meses mínimos de entrenamiento, solo hay **un origen de evaluación** (octubre de 2025), con noviembre y diciembre como objetivos. Cada método se evalúa en seis grupos y dos horizontes. Esto no es evidencia suficiente para declarar un modelo ganador definitivo. El perfil nacional se reconstruye con años completos anteriores al origen para evitar información futura.

In [ ]:
fig = px.bar(resumen, x="metodo", y="MAE", title="MAE por método (dos meses consecutivos)")
fig.show()

In [ ]:
fig = px.bar(detalle.groupby(["metodo","horizonte"],as_index=False).error.apply(lambda s: s.abs().mean()).rename(columns={"error":"MAE"}), x="metodo",y="MAE",color="horizonte",barmode="group",title="Error por horizonte")
fig.show()

In [ ]:
display(detalle[["origen","fecha","horizonte","subdestino","segmento","metodo","observado","estimado","error"]].head(16))

,origen,fecha,horizonte,subdestino,segmento,metodo,observado,estimado,error
0,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,mayor a 1 dia,media_movil,33440.431701,29638.967446,-3801.464255
1,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,mayor a 1 dia,persistencia,33440.431701,30808.712989,-2631.718712
2,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,mayor a 1 dia,reparto_mediano,33440.431701,37728.254499,4287.822798
3,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,mayor a 1 dia,reparto_ultimo,33440.431701,36214.498094,2774.066393
4,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,menor a 1 dia,media_movil,58743.175479,56506.768087,-2236.407391
5,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,menor a 1 dia,persistencia,58743.175479,60513.809494,1770.634015
6,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,menor a 1 dia,reparto_mediano,58743.175479,71748.712295,13005.536816
7,2025-10-01,2025-11-01,1,El Remate Y Alrededores Lago,menor a 1 dia,reparto_ultimo,58743.175479,71131.735992,12388.560513
8,2025-10-01,2025-11-01,1,Isla De Flores,mayor a 1 dia,media_movil,37510.399442,35317.452988,-2192.946455
9,2025-10-01,2025-11-01,1,Isla De Flores,mayor a 1 dia,persistencia,37510.399442,36040.106153,-1470.293289


## Conclusión y siguientes pasos

La comparación solo cubre noviembre y diciembre de 2025. Para pronosticar enero-febrero 2026 no podemos usar observaciones de esos meses. Se recomienda ampliar la validación con ventanas independientes si se dispone de más historia mensual del destino, y mantener la regla de aceptación vigente hasta contar con evidencia más robusta.

In [ ]:

import pandas as pd

mov = pd.read_csv(
    DATA / "movilidad_mensual_peten.csv",
    parse_dates=["fecha"]
)

# Comprobar cobertura por subdestino y segmento
cobertura = (
    mov.groupby(["subdestino", "segmento"])
    .agg(
        primer_mes=("fecha", "min"),
        ultimo_mes=("fecha", "max"),
        meses_disponibles=("fecha", "nunique")
    )
    .reset_index()
)

display(cobertura)

# Meses comunes a todos los grupos
grupos = mov[["subdestino", "segmento"]].drop_duplicates()
total_grupos = len(grupos)

meses = (
    mov.groupby("fecha")
    .agg(grupos_presentes=("subdestino", "size"))
    .reset_index()
    .sort_values("fecha")
)

meses["cobertura_completa"] = (
    meses["grupos_presentes"] == total_grupos
)

display(meses)


,subdestino,segmento,primer_mes,ultimo_mes,meses_disponibles
0,El Remate Y Alrededores Lago,mayor a 1 dia,2025-09-01,2026-08-01,10
1,El Remate Y Alrededores Lago,menor a 1 dia,2025-09-01,2026-08-01,10
2,Isla De Flores,mayor a 1 dia,2025-09-01,2026-08-01,10
3,Isla De Flores,menor a 1 dia,2025-09-01,2026-08-01,10
4,Mundo Maya Aeropuerto,mayor a 1 dia,2025-09-01,2026-08-01,10
5,Mundo Maya Aeropuerto,menor a 1 dia,2025-09-01,2026-08-01,10


,fecha,grupos_presentes,cobertura_completa
0,2025-09-01,6,True
1,2025-10-01,6,True
2,2025-11-01,6,True
3,2025-12-01,6,True
4,2026-03-01,6,True
5,2026-04-01,6,True
6,2026-05-01,6,True
7,2026-06-01,6,True
8,2026-07-01,6,True
9,2026-08-01,6,True
